# __Dream interpretation RAG__

## __Installs & imports__

In [1]:
#! pip install chromadb #sentence-transformers

In [2]:
#! pip install numpy==1.26.4 torch==2.2.2 transformers==4.57.6

In [1]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import re
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import string

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/lesfabs/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', None)


## __Data preparation__

In [4]:
url = '/Users/lesfabs/code/MartynaC/dreamscope/dreamscope_backend/data/dream_symbols_clean_v5.csv'
data = pd.read_csv(url)
data.head()

,symbol_clean,slug,context,context_clean,meaning_clean
0,m,m,"To see the letter ""M"" in your dream",to see the letter m in your dream,suggests that there is something that you are keeping silent about perhaps you have been sworn to secrecy alternatively the dream may imply mmmmm your subconscious mind is hungering for knowledge or information as a roman numeral it could represent the number 1000
1,m&m's,mms,To see or eat M&M's in your dream,to see or eat mms in your dream,symbolizes lifes small but sweet rewards more directly mm may represent the initials of two people in your waking life
2,m&m's,mms,To dream that you are a giant talking M&M,to dream that you are a giant talking mm,suggests that you feel you are being mislead or taking advantaged of perhaps you feel that you are being someone you are not in order to please others
3,macadamize,macadamize,"To see, walk or travel on a macadamized road in your dream",to see walk or travel on a macadamized road in your dream,suggests that you are standing on solid ground you have laid out a solid groundwork for much success in your life
4,macaroni,macaroni,To dream that you are eating macaroni,to dream that you are eating macaroni,symbolizes comfort and ease the dream may be trying to bring you back to a time where things were much simpler


In [5]:
data.context_clean.isnull().sum()

np.int64(7)

In [6]:
data.loc[data['context_clean'].isnull(), 'context_clean']

608     NaN
775     NaN
1711    NaN
2333    NaN
3624    NaN
4001    NaN
6680    NaN
Name: context_clean, dtype: object

In [7]:
data.loc[data['context_clean'].isnull(), 'slug']

608           ace
775         amber
1711    disappear
2333          pin
3624      bowling
4001       school
6680    crocodile
Name: slug, dtype: object

In [8]:
data[data['context_clean'].isnull()]

,symbol_clean,slug,context,context_clean,meaning_clean
608,ace,ace,NaN,NaN,alternatively the dream may be a metaphor that you are an ace or the dream is a pun on acing a test
775,amber,amber,NaN,NaN,alternatively to see amber in your dream represents resurrection something in your past will prove to be extremely important to your future it could mean a situation or relationship in your waking life that was once lively is now nonexistent you feel trapped or that your life is too rigid and inflexible you need to change your outdated way of thinking and old ideas
1711,disappear,disappear,NaN,NaN,alternatively to dream that someone is disappearing suggests that you may not have given sufficient attention to those aspectsqualities of that person within your own self have you lost touch with some aspect of yourself if your lover is disappearing then it suggests that your love or interest for them is fading
2333,pin,pin,NaN,NaN,alternatively to see pins in your dream er to feeling of being trapped or immobilized as exemplified by the phrase being pinned down
3624,bowling,bowling,NaN,NaN,alternatively bowling and bowling alleys may also be a metaphor for sexual conquest consider the sexual innuendos that are at play in the bowling alley the pins and bowling balls can be viewed as masculine symbols the pin deck is symbolic of the womb or vagina as is with any dark receptacle like caves bowls containers etc
4001,school,school,NaN,NaN,alternatively a dream that takes place in school may be a metaphor for the lessons that you are learning from your waking life you may be going through a spiritual learning experience
6680,crocodile,crocodile,NaN,NaN,alternatively the crocodile may be an aspect of yourself and your aggressive and snappy attitude or it may reveal that you are being insincere displaying false emotions and shedding crocodile tears


In [9]:
# Replace NaNs in  context_clean with the value of the slug column
data.loc[data['context_clean'].isnull(), 'context_clean'] = data.loc[data['context_clean'].isnull(), 'slug']
data.context_clean.isnull().sum()

np.int64(0)

In [10]:
data[data.symbol_clean.isin(['ace', 'amber', 'disappear'])]

,symbol_clean,slug,context,context_clean,meaning_clean
607,ace,ace,To see the ace in the deck of cards,to see the ace in the deck of cards,suggests ambiguity in your life you need some clarity in particular the ace of hearts means that you are involved in some love affair if you see the ace of spades in your dream then it means that you are involved in a scandal if you see the ace of diamonds then it symbolizes your legacy or reputation and if you see the ace of clubs then it indicates that you will be involved in some legal matter
608,ace,ace,NaN,ace,alternatively the dream may be a metaphor that you are an ace or the dream is a pun on acing a test
774,amber,amber,To see amber in your dream,to see amber in your dream,symbolizes the sun and positive energy amber is said to have natural healing power some believe that the amber can heal sore eyes sprained limbs or arthritis thus the amber in your dream could mean you need to be healed in some way
775,amber,amber,NaN,amber,alternatively to see amber in your dream represents resurrection something in your past will prove to be extremely important to your future it could mean a situation or relationship in your waking life that was once lively is now nonexistent you feel trapped or that your life is too rigid and inflexible you need to change your outdated way of thinking and old ideas
1710,disappear,disappear,To dream that people or objects are disappearing right before your eyes,to dream that people or objects are disappearing right before your eyes,signify your anxieties and insecurities over the notion that loved ones might disappear out of your life you feel that you cannot depend on anyone and that you will end up alone you need to work on your selfimage and selfesteem
1711,disappear,disappear,NaN,disappear,alternatively to dream that someone is disappearing suggests that you may not have given sufficient attention to those aspectsqualities of that person within your own self have you lost touch with some aspect of yourself if your lover is disappearing then it suggests that your love or interest for them is fading
1712,disappear,disappear,To dream that you are disappearing from others,to dream that you are disappearing from others,means that you feel you are being overlooked you are not being noticed or recognized for what is important to you alternatively the dream indicates that you are trying to withdraw from the realities of life


In [13]:
data.shape

(7798, 5)

In [11]:
# Create chunks
data['chunk'] = data['context_clean'] + " " + data['meaning_clean']
data.sample(20)
data[data['slug']=='fire']

,symbol_clean,slug,context,context_clean,meaning_clean,chunk
5530,fire,fire,"Depending on the context of your dream, to see fire in your dream can",depending on the context of your dream to see fire in your dream can,symbolize destruction passion desire illumination purification transformation enlightenment or anger if you are not afraid of the fire and it is under control or contained in one area then it is a symbol of your own internal fire and inner transformation something old is passing and something new is entering into your life your thoughts and views are changing if the fire is encircling you and someone else then it signifies your bond to that person the two of you share something significant the dream may be a metaphor for someone who is fiery it can also represent your drive motivation and creative energy alternatively the dream may be warning you of some dangerous or risky activities you are playing with fire,depending on the context of your dream to see fire in your dream can symbolize destruction passion desire illumination purification transformation enlightenment or anger if you are not afraid of the fire and it is under control or contained in one area then it is a symbol of your own internal fire and inner transformation something old is passing and something new is entering into your life your thoughts and views are changing if the fire is encircling you and someone else then it signifies your bond to that person the two of you share something significant the dream may be a metaphor for someone who is fiery it can also represent your drive motivation and creative energy alternatively the dream may be warning you of some dangerous or risky activities you are playing with fire
5531,fire,fire,To dream that you are being burned by fire,to dream that you are being burned by fire,indicates that your temper is getting out of control some issue or situation is burning you up inside,to dream that you are being burned by fire indicates that your temper is getting out of control some issue or situation is burning you up inside
5532,fire,fire,To dream that a house is on fire,to dream that a house is on fire,indicates that you need to undergo some transformation if you have recurring dreams of your family house on fire then it suggests that you are still not ready for the change or that you are fighting against the change alternatively it highlights passion and the love of those around you,to dream that a house is on fire indicates that you need to undergo some transformation if you have recurring dreams of your family house on fire then it suggests that you are still not ready for the change or that you are fighting against the change alternatively it highlights passion and the love of those around you
5533,fire,fire,To dream that you put out a fire,to dream that you put out a fire,signifies that you will overcome your obstacles in your life through much work and effort if you are setting a fire to something or even to yourself then it indicates that you are undergoing some great distress you are at the brink of desperation and want to destroy something or some aspect of yourself,to dream that you put out a fire signifies that you will overcome your obstacles in your life through much work and effort if you are setting a fire to something or even to yourself then it indicates that you are undergoing some great distress you are at the brink of desperation and want to destroy something or some aspect of yourself
5534,fire,fire,To dream that you can start fire with your hands,to dream that you can start fire with your hands,represents anger that you are trying to repress you have difficulty expressing yourself and tend to overreact,to dream that you can start fire with your hands represents anger that you are trying to repress you have difficulty expressing yourself and tend to overreact
5535,fire,fire,To dream that you can bend fire,to dream that you can bend fire,ers to your ability to control your anger,to dream that y

In [12]:
data[data['slug']=='laughing']

,symbol_clean,slug,context,context_clean,meaning_clean,chunk
2787,laughing,laughing,To hear laughing or dream that you are laughing,to hear laughing or dream that you are laughing,suggests that you need to lighten up and let go of your problems dont put so much pressure on yourself laughing is also a sign of joyous release and pleasure if you are being laughed at then it indicates your insecurities and fears of not being accepted,to hear laughing or dream that you are laughing suggests that you need to lighten up and let go of your problems dont put so much pressure on yourself laughing is also a sign of joyous release and pleasure if you are being laughed at then it indicates your insecurities and fears of not being accepted
2788,laughing,laughing,"To hear evil, demonic laughing in your dream",to hear evil demonic laughing in your dream,represents feelings of humiliation andor helplessness you feel that someone is working against you,to hear evil demonic laughing in your dream represents feelings of humiliation andor helplessness you feel that someone is working against you


In [13]:
data = data[data['context_clean'] != data['meaning_clean']].reset_index(drop=True)

In [14]:
data.shape

(7551, 6)

In [15]:
#chunk length
data['chunk'].map(lambda x: len(x.split(' ')))

0       51
1       28
2       37
3       33
4       28
        ..
7546    17
7547    29
7548    16
7549    20
7550    23
Name: chunk, Length: 7551, dtype: int64

In [16]:
data['chunk'].map(lambda x: len(x.split(' '))).max(), data['chunk'].map(lambda x: len(x.split(' '))).mean()

(np.int64(272), np.float64(36.33624685472123))

## __MVP__

### __Embedding__

In [17]:
sentence_transformer_model = SentenceTransformer("all-mpnet-base-v2", device = 'cpu')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
data['chunk'][0]

'to see the letter m in your dream suggests that there is something that you are keeping silent about perhaps you have been sworn to secrecy alternatively the dream may imply mmmmm your subconscious mind is hungering for knowledge or information as a roman numeral it could represent the number 1000'

In [19]:
# One chunk
tryout_vector = sentence_transformer_model.encode(data['chunk'][0])
tryout_vector.shape

(768,)

In [20]:
# All chunks

embeddings = sentence_transformer_model.encode(data['chunk'].to_list(), show_progress_bar=True)


Batches:   0%|          | 0/236 [00:00<?, ?it/s]

In [21]:
embeddings.shape

(7551, 768)

In [22]:
embeddings_context_clean = sentence_transformer_model.encode(data['context_clean'].to_list(), show_progress_bar=True)
embeddings_context_clean.shape

Batches:   0%|          | 0/236 [00:00<?, ?it/s]

(7551, 768)

### __Setting up and querying collection__

In [23]:
# Initializing collection - slightly different from the method seen in previous challenges using LangChain
# -> Here it's native ChromaDB, without an intermediary framework
client = chromadb.Client()

In [15]:
# To delete the collection if needed
#client.delete_collection("dream_symbols")

In [24]:
collection = client.create_collection("dream_symbols")

In [32]:
# Adding and indexing chunks

collection.add(
    documents=data['chunk'].tolist()[:5000],
    embeddings=embeddings[:5000],
    ids=[str(i) for i in range(5000)]
)

In [33]:
# We do it in two batches because of the limited batch size

collection.add(
    documents=data['chunk'].tolist()[5000:],
    embeddings=embeddings[5000:],
    ids=[str(i) for i in range(5000, 7551)]
)

In [34]:
collection.count()

7551

In [29]:
dreamer_text = "I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I open the door and fall back in my bed."

In [30]:
searched_vector = sentence_transformer_model.encode(dreamer_text)
searched_vector.shape

(768,)

In [35]:
result=collection.query(
    query_embeddings=[searched_vector],
    n_results=6
)
result['documents']

[['to dream that you are lost in a forest indicates that you are searching through your subconscious for a better understanding of yourself',
  'to dream that you are in or walking through the forest signifies a transitional phase follow your instincts alternatively it indicates that you want to escape to a simpler way of life you are feeling weighed down by the demands of your life',
  'to see or dream that you are in a wood cabin indicates that you will succeed via your own means it suggests that you are selfreliant and independent yet still remain humble you per the simpler things in life',
  'to discover secret passageways in your dream parallel to something new andor exciting that is occurring in your waking life it ers to a new opportunity a new relationship or a new attitude toward life if you wake up before fully exploring these passageways then it suggest that you may not know how to take advantage of these opportunities or how to move forward with a relationship perhaps the n

#### Embedding only the context clean

In [25]:
collection_context_clean = client.create_collection("dream_symbols_context_clean")

In [26]:
collection_context_clean.add(
    documents=data['context_clean'].tolist()[:5000],
    embeddings=embeddings_context_clean[:5000],
    ids=[str(i) for i in range(5000)]
)

In [27]:
collection_context_clean.add(
    documents=data['context_clean'].tolist()[5000:],
    embeddings=embeddings_context_clean[5000:],
    ids=[str(i) for i in range(5000, 7551)]
)

In [28]:
collection_context_clean.count()

7551

In [36]:
result=collection_context_clean.query(
    query_embeddings=[searched_vector],
    n_results=6
)
result['documents']

[['to dream that you are lost in a forest',
  'to see a hallway in your dream',
  'to dream that you are entering through a door',
  'to dream that you fall into a hole',
  'to dream that you are lost in a park',
  'to see dracula in your dream']]

In [37]:
result

{'ids': [['5530', '7162', '1777', '7389', '2029', '1811']],
 'embeddings': None,
 'documents': [['to dream that you are lost in a forest',
   'to see a hallway in your dream',
   'to dream that you are entering through a door',
   'to dream that you fall into a hole',
   'to dream that you are lost in a park',
   'to see dracula in your dream']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None, None, None, None]],
 'distances': [[0.8376301527023315,
   0.8791440725326538,
   0.8812369108200073,
   0.8866716027259827,
   0.8973263502120972,
   0.8992968797683716]]}

In [38]:
dreamer_text

"I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I open the door and fall back in my bed."

##### Adding a reranker

In [39]:
from sentence_transformers import CrossEncoder

#reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
reranker = CrossEncoder("BAAI/bge-reranker-base")

searched_vector = sentence_transformer_model.encode(dreamer_text)
result = collection_context_clean.query(query_embeddings=searched_vector, n_results=20)

pairs = [(dreamer_text, doc) for doc in result['documents'][0]]
pairs

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[("I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I open the door and fall back in my bed.",
  'to dream that you are lost in a forest'),
 ("I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I open the door and fall back in my bed.",
  'to see a hallway in your dream'),
 ("I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I open the door and fall back in my bed.",
  'to dream that you are entering through a door'),
 ("I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I ope

In [40]:
scores = reranker.predict(pairs)
scores

array([9.8787540e-01, 1.4111465e-03, 1.0454042e-01, 5.6863755e-05,
       5.5077061e-02, 7.9308025e-05, 3.6932706e-04, 2.8143949e-03,
       7.3614574e-05, 5.8445108e-01, 1.9980539e-02, 2.0926543e-04,
       4.9401483e-01, 2.7463190e-02, 2.5993371e-01, 3.7289759e-05,
       3.7331603e-05, 1.7275346e-02, 1.4803325e-02, 1.4803325e-02],
      dtype=float32)

In [42]:
# Return top n_final by re-ranked score
ranked = sorted(zip(result['documents'][0], scores), key=lambda x: x[1], reverse=True)
ranked

[('to dream that you are lost in a forest', np.float32(0.9878754)),
 ('to dream that you are in or walking through the forest',
  np.float32(0.5844511)),
 ('to see an opened door in your dream', np.float32(0.49401483)),
 ('to find or see a trapdoor in your dream', np.float32(0.2599337)),
 ('to dream that you are entering through a door', np.float32(0.10454042)),
 ('to dream that you are lost in a park', np.float32(0.05507706)),
 ('to see a doorknob in your dream', np.float32(0.02746319)),
 ('to see your bed in your dream', np.float32(0.019980539)),
 ('to see a tree stump in your dream', np.float32(0.017275346)),
 ('to see a castle door in your dream', np.float32(0.014803325)),
 ('to see a castle door in your dream', np.float32(0.014803325)),
 ('to see a falling maple tree in your dream', np.float32(0.002814395)),
 ('to see a hallway in your dream', np.float32(0.0014111465)),
 ('dreaming that you are in a dark room with no doors',
  np.float32(0.00036932706)),
 ('to dream that you are i

In [43]:
def reformulate_query(dream_text):
    sentences = sent_tokenize(dream_text)
    return ["to dream that " + s.lower().strip(".") for s in sentences]

reformulated_queries = reformulate_query(dreamer_text)
reformulated_queries


['to dream that i am eating strawberry jam in a deep forest',
 "to dream that then i realize i'm lost and start being afraid",
 'to dream that i search my way through different landscapes before finding a door embedded in a tree',
 'to dream that i open the door and fall back in my bed']

In [44]:

for sentence in reformulated_queries:
    searched_vector = sentence_transformer_model.encode(sentence)
    result = collection_context_clean.query(query_embeddings=searched_vector, n_results=20)

    pairs = [(dreamer_text, doc) for doc in result['documents'][0]]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(result['documents'][0], scores), key=lambda x: x[1], reverse=True)
    print(ranked[:3])


[('to dream that you are lost in a forest', np.float32(0.9878754)), ('to see or eat strawberries in your dream', np.float32(0.71464264)), ('to see or eat berries in your dream', np.float32(0.06653912))]
[('to dream that you are lost in a forest', np.float32(0.9878754)), ('to dream that you are lost', np.float32(0.5889848)), ('to dream that someone else is lost', np.float32(0.1898073))]
[('to dream that you are lost in a forest', np.float32(0.9878754)), ('to dream that you are in or walking through the forest', np.float32(0.5844511)), ('to see an opened door in your dream', np.float32(0.49401483))]
[('to see an opened door in your dream', np.float32(0.49401483)), ('to dream that the doors to the patio are opened', np.float32(0.25713807)), ('to dream that you are entering through a door', np.float32(0.10454042))]


##### ChromaDB with cosine search

In [45]:
collection_context_clean_cosine = client.create_collection("dream_symbols_context_clean_cosine",
                                                            metadata={"hnsw:space": "cosine"})

In [46]:
collection_context_clean_cosine.add(
    documents=data['context_clean'].tolist()[:5000],
    embeddings=embeddings_context_clean[:5000],
    ids=[str(i) for i in range(5000)]
)

collection_context_clean_cosine.add(
    documents=data['context_clean'].tolist()[5000:],
    embeddings=embeddings_context_clean[5000:],
    ids=[str(i) for i in range(5000, len(data))]
)

In [47]:
result_cosine = collection_context_clean_cosine.query(
    query_embeddings=[searched_vector],
    n_results=6
)
result_cosine['documents']

[['to see your bed in your dream',
  'to dream that you are entering through a door',
  'to see an opened door in your dream',
  'to dream that the drawers of the file cabinet are wide open',
  'to dream that you are locking the door',
  'to dream that you are opening the garage door']]

#### Embde chunk but with metadata

In [77]:
#client.delete_collection("dream_symbols_metadata")


In [49]:
collection_context_metadata = client.create_collection("dream_symbols_metadata")

In [50]:
metadatas = data[['slug', 'meaning_clean']].to_dict(orient='records')
metadatas

[{'slug': 'm',
  'meaning_clean': 'suggests that there is something that you are keeping silent about perhaps you have been sworn to secrecy alternatively the dream may imply mmmmm your subconscious mind is hungering for knowledge or information as a roman numeral it could represent the number 1000'},
 {'slug': 'mms',
  'meaning_clean': 'symbolizes lifes small but sweet rewards more directly mm may represent the initials of two people in your waking life'},
 {'slug': 'mms',
  'meaning_clean': 'suggests that you feel you are being mislead or taking advantaged of perhaps you feel that you are being someone you are not in order to please others'},
 {'slug': 'macadamize',
  'meaning_clean': 'suggests that you are standing on solid ground you have laid out a solid groundwork for much success in your life'},
 {'slug': 'macaroni',
  'meaning_clean': 'symbolizes comfort and ease the dream may be trying to bring you back to a time where things were much simpler'},
 {'slug': 'macaroon',
  'meani

In [51]:
data.columns

Index(['symbol_clean', 'slug', 'context', 'context_clean', 'meaning_clean',
       'chunk'],
      dtype='object')

In [52]:
collection_clean_context_metadata.add(
    documents=data['context_clean'].tolist()[:5000],
    embeddings=embeddings[:5000],
    metadatas=metadatas[:5000],
    ids=[str(i) for i in range(5000)]
)

collection_clean_context_metadata.add(
    documents=data['context_clean'].tolist()[5000:],
    embeddings=embeddings[5000:],
    metadatas=metadatas[5000:],
    ids=[str(i) for i in range(5000, len(data))]
)

In [53]:
dreamer_text

"I am eating strawberry jam in a deep forest. Then I realize I'm lost and start being afraid. I search my way through different landscapes before finding a door embedded in a tree. I open the door and fall back in my bed."

In [56]:
! pip install spacy
! python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 10.3 MB/s  0:00:00a 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 658.0/658.0 kB 9.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.4/815.4 kB 6.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 8.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [spacy]m━━ 14/15 [spacy][srsly]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 7.9 MB/s  0:00:018.1 MB/s eta 0:00:01:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [57]:
import spacy
nlp = spacy.load("en_core_web_sm")


In [58]:

def extract_keywords(text):
    doc = nlp(text)
    return [token.lemma_ for token in doc if token.pos_ in ("VERB","NOUN", "ADJ") and not token.is_stop]

key_words = extract_keywords(dreamer_text)
key_words

['eat',
 'strawberry',
 'jam',
 'deep',
 'forest',
 'realize',
 'lose',
 'start',
 'afraid',
 'search',
 'way',
 'different',
 'landscape',
 'find',
 'door',
 'embed',
 'tree',
 'open',
 'door',
 'fall',
 'bed']

In [59]:
result_metadata = collection_clean_context_metadata.query(
    query_embeddings=searched_vector,
    n_results=6,
    where={"slug": {"$in": key_words}}
)
result_metadata['documents']

[['to dream that you are locking the door',
  'to dream that you are entering through a door',
  'in particular seeing the front door in your dream',
  'to see an opened door in your dream',
  'to dream that you are searching for a bed',
  'to see your bed in your dream']]

In [ ]:
result_metadata = collection_chunk_metadata.query(
    query_embeddings=searched_vector,
    n_results=6,
    where={"slug": {"$in": key_words}}
)
result_metadata

In [110]:
result_metadata['documents']

[['to dream that you are in a cage indicates that you are experiencing inhibitions and powerlessness in some areas of your life you are feeling restricted confined and restrained in a current relationship or business deal somebody may be keeping a short leash on you where you are lacking the freedom to act independently',
  'to dream that you are trapped or caught in a trap suggests that you are feeling confined and restricted in your job career health or a personal relationship you may be in a rut and are tired of the same daily monotony',
  'to dream that you are putting a wild animal into a cage signifies that you will succeed in overcoming your rivals and fears it is also symbolic of your ability to control you animalistic rages and anger',
  'to dream that your computer has a mind of its own denotes anxiety about technology and loss of control you are feeling overwhelmed and that you are at the mercy of another',
  'to see a computer in your dream symbolizes technology information

### __Compiling query process in a dedicated function__ 

In [61]:
# Compile code in a dedicated function

def interpret(dream='',n_results=3):
    searched_vector = sentence_transformer_model.encode(dream)
    result=collection.query(
    searched_vector,
    n_results=n_results
    )
    return result['documents']

interpret(dreamer_text)

[['to dream that you are lost in a forest indicates that you are searching through your subconscious for a better understanding of yourself',
  'to dream that you are in or walking through the forest signifies a transitional phase follow your instincts alternatively it indicates that you want to escape to a simpler way of life you are feeling weighed down by the demands of your life',
  'to see or dream that you are in a wood cabin indicates that you will succeed via your own means it suggests that you are selfreliant and independent yet still remain humble you per the simpler things in life']]

## __Trying to fine-tune the querying process__

### __Extracting keywords - Simple version__

In [63]:
# Initialize Model

#tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')
#model = AutoModelForCausalLM.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

# Initialize pipeline

pipe = pipeline(
    "text-generation",
    model='microsoft/Phi-3-mini-4k-instruct',
    device='cpu'
    #tokenizer=tokenizer,
    )


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [64]:
# Define function to extract keywords

def extract_keywords(dreamer_text, n_keywords):
    # define prompt template
    prompt = f'You process a text input by extracting the {n_keywords} most significant keywords, ranked by significance. Expected format : a Python list of singular words, without any additional text'
    messages = [
        {'role': 'system', 'content': prompt},
        {'role': 'user', 'content': dreamer_text}
    ]

    # query model
    output = pipe(messages)
    return output[0]['generated_text'][-1]['content']


In [65]:
result = extract_keywords(dreamer_text, 6)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [32]:
result

' ["cage", "slimy", "green", "strategy", "consultant", "computer"]'

In [33]:
keywords = re.findall(r"['\"]([^'\"]+)['\"]", result)


In [34]:
keywords

['cage', 'slimy', 'green', 'strategy', 'consultant', 'computer']

In [36]:
for word in keywords:
    searched_vector = sentence_transformer_model.encode(word)
    result = collection.query(
        searched_vector,
        n_results=1
    )
    print(word, result['documents'][0])

cage ['to watch a cage fight represents conflicting ideas or beliefs']
slimy ['to see or feel slime in your dream represents your inability to place your trust in someone the dream may be a metaphor for some one is a slimeball']
green ['the heart chakra is depicted by a green color and symbolizes pure and divine love for everyone and everything around you']
strategy ['to play foosball in your dream ers to a situation where you need to respond or act quickly you can not afford to lose site of your goal']
consultant ['to see a client in your dream ers to your position of power consider the position that you are in and the significance of the services that you are providing if you dream that you are a client then it means that you are looking for some sort of approval and validation of your ideas']
computer ['to see a computer in your dream symbolizes technology information and modern life new areas of opportunities are being opened to you alternatively computers represent a lack of indiv

### __Extracting keywords - Advanced version with preprocessing__

In [ ]:
# Improve function to extract keywords -> Preprocess dream text

stop_words = set(stopwords.words('english'))

def preprocess(dream_text):
    # remove punctuation
    for punctuation in string.punctuation:
        dream_text = dream_text.replace(punctuation, ' ')

    # lowercase
    dream_text = dream_text.lower()

    # remove stopwords
    tokenized_text = word_tokenize(dream_text)
    clean_text = [word for word in tokenized_text if not word in stop_words]

    # lemmatize
    lemmatizer = WordNetLemmatizer()
    lemmatized = [lemmatizer.lemmatize(word) for word in clean_text]
    lemmatized_string = " ".join(lemmatized)

    return lemmatized_string


In [38]:
preprocessed_dream = preprocess(dreamer_text)
preprocessed_dream

'trapped cage slimy green strategy consultant eating computer'

In [39]:
result = extract_keywords(preprocessed_dream, 6)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [40]:
keywords = re.findall(r"['\"]([^'\"]+)['\"]", result)
keywords

['strategy', 'consultant', 'cage', 'eating', 'slimy', 'green']

In [41]:
for word in keywords:
    searched_vector = sentence_transformer_model.encode(word)
    result=collection.query(
        searched_vector,
        n_results=1
    )
    print(word, result['documents'][0])

strategy ['to play foosball in your dream ers to a situation where you need to respond or act quickly you can not afford to lose site of your goal']
consultant ['to see a client in your dream ers to your position of power consider the position that you are in and the significance of the services that you are providing if you dream that you are a client then it means that you are looking for some sort of approval and validation of your ideas']
cage ['to watch a cage fight represents conflicting ideas or beliefs']
eating ['to see someone eating with a fork denotes that all your current worries will be cleared up through the help of a friend']
slimy ['to see or feel slime in your dream represents your inability to place your trust in someone the dream may be a metaphor for some one is a slimeball']
green ['the heart chakra is depicted by a green color and symbolizes pure and divine love for everyone and everything around you']


__Problem with the approach :__ Working with keywords loses the context, so that when symbols have multiple context-dependant entries in the RAG it's pure chance whether we're going to get the right one. For instance, "Fire" as a keyword does not discriminate between "starting a fire", "being on fire", "fire truck"... 

In [ ]:
## TODO
# Try BerTopic https://maartengr.github.io/BERTopic/index.html // First out of the box, then trained on the symbols dictionary.
# Also try Named Entity Recognition (gleaner) to extract the most significant words from the text.
# Generate final response with LLM

### __Querying the RAG - Alternate approach with embedding by sentences__

In [61]:
def tokenize_sentences(dreamer_text):
    sentences = sent_tokenize(dreamer_text)
    return sentences

In [62]:
sentences = tokenize_sentences(dreamer_text)

In [64]:
# Compile code in a dedicated function

def interpret_sentences(dream_sentences, n_results=3):
    interpretations = []
    for sentence in dream_sentences :
        searched_vector = sentence_transformer_model.encode(sentence)
        result=collection.query(
            searched_vector,
            n_results=n_results
        )
        interpretations.append(result['documents'][0])
    return interpretations

interpret_sentences(sentences)

[['to dream that you are in a cage indicates that you are experiencing inhibitions and powerlessness in some areas of your life you are feeling restricted confined and restrained in a current relationship or business deal somebody may be keeping a short leash on you where you are lacking the freedom to act independently',
  'to dream that you are trapped or caught in a trap suggests that you are feeling confined and restricted in your job career health or a personal relationship you may be in a rut and are tired of the same daily monotony',
  'to dream that you kill a leopard ers to success in your projects'],
 ['to dream that your hard drive crashed indicates that you are being overwhelmed with information perhaps something is taking a emotional toll on you alternatively the dream may be a metaphor that your unusually strong will and drive will set you on a crash course',
  'to dream that a computer has a virus or has crashed suggests that something in your life that is out of control

### __Combining the approaches__

In [2]:
# Dream text
dreamer_text = "I am very angry at my kids and I am shouting about something. My kids start crying and I suddenly feel super sorry. I look at my hands and they have turned a vivid green. I realize that I am turning into a monster"

In [3]:
# Load and prepare data
url = '/Users/lesfabs/code/MartynaC/dreamscope/dreamscope_backend/data/dream_symbols_clean_v5.csv'
data = pd.read_csv(url)

# Replace NaNs in  context_clean with the value of the slug column
data.loc[data['context_clean'].isnull(), 'context_clean'] = data.loc[data['context_clean'].isnull(), 'slug']
data = data[data['context_clean'] != data['meaning_clean']].reset_index(drop=True)

In [5]:
#client.delete_collection("dream_symbols_metadata")

In [6]:
# Initialize sentence transformer and define context embeddings
sentence_transformer_model = SentenceTransformer("all-mpnet-base-v2", device = 'cpu')
embeddings_context_clean = sentence_transformer_model.encode(data['context_clean'].to_list(), show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/236 [00:00<?, ?it/s]

In [7]:
# Set up the RAG with medadata
client = chromadb.Client()

# Create and feed collection with context and metadata
collection_context_metadata = client.create_collection("dream_symbols_metadata")
metadatas = data[['slug', 'meaning_clean']].to_dict(orient='records')

collection_context_metadata.add(
    documents=data['context_clean'].tolist()[:5000],
    embeddings=embeddings_context_clean[:5000],
    metadatas=metadatas[:5000],
    ids=[str(i) for i in range(5000)]
)

collection_context_metadata.add(
    documents=data['context_clean'].tolist()[5000:],
    embeddings=embeddings_context_clean[5000:],
    metadatas=metadatas[5000:],
    ids=[str(i) for i in range(5000, len(data))]
)

In [8]:
# Preprocess dream text

# Divide into sentences
def tokenize_sentences(dreamer_text):
    sentences = sent_tokenize(dreamer_text)
    return sentences

sentences = tokenize_sentences(dreamer_text)

# Embed dream sentences
searched_vectors = sentence_transformer_model.encode(sentences)

In [9]:
sentences

['I am very angry at my kids and I am shouting about something.',
 'My kids start crying and I suddenly feel super sorry.',
 'I look at my hands and they have turned a vivid green.',
 'I realize that I am turning into a monster']

In [15]:
# Query the vector db
result_metadata = collection_context_metadata.query(
query_embeddings=searched_vectors,
n_results=6,
)

# Generate context - interpretation tuples and removing duplicates
flat_results = [
    (context, interpretation['meaning_clean'])
    for contexts, interpretations in zip(result_metadata['documents'], result_metadata['metadatas'])
    for context, interpretation in zip(contexts, interpretations)
]
flat_results_unique = set(flat_results)
flat_results_unique

{('anger',
  'being angry in your dream may have been carried over from your waking life dreams function as a safe outlet where you can express your strong andor negative emotions you are suppressing your anger and aggression instead of consciously acknowledging them'),
 ('dreaming of horrific things',
  'implies that you are numb by the things around you you are becoming too indifferent'),
 ('if you dream that you are scolding someone',
  'then it means that there is some resentment or unexpressed anger that you need to confront'),
 ('if you dream that your toenails are dark and discolored',
  'then it indicates regret in some decision you made or in some action that you took'),
 ('to dream of a crying baby',
  'symbolizes a part of yourself that is deprived of attention and needs to be nurtured alternatively it represents your unfulfilled goals and a sense of lacking in your life if you dream that a baby is neglected then it suggests that you are not paying enough attention to yourse

In [16]:
# Rerank results

# Initialize reranker
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-base")

# Define symbol context-dream text pairs
pairs = [(dreamer_text, symbol[0]) for symbol in flat_results]

# Rerank according to results relevance
scores = reranker.predict(pairs)

# Trier en gardant les métadonnées
ranked = sorted(zip(flat_results, scores), key=lambda x: x[1], reverse=True)

# Extract list of ranked interpretations
interpretations = [symbol[0][0] + ' ' + symbol[0][1] for symbol in ranked]
interpretations

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['anger being angry in your dream may have been carried over from your waking life dreams function as a safe outlet where you can express your strong andor negative emotions you are suppressing your anger and aggression instead of consciously acknowledging them',
 'to turn into a monster in your dream suggests that you are becoming someone who you are ashamed of or someone who you longer recognized or the dream may also be showing your true character perhaps you need to change your attitude or ways',
 'to dream that you are holding or expressing anger symbolizes frustrations and disappointments in your self you tend to repress your negative emotions or project your anger onto others you need to look within yourself',
 'to dream that you kill a monster means that you will successfully overcome your rivals and advance to a higher position',
 'to see someone else crying in your dream may be a projection of your own feelings onto someone else if you do not cry in your waking life then seei

In [17]:
interpretations

['anger being angry in your dream may have been carried over from your waking life dreams function as a safe outlet where you can express your strong andor negative emotions you are suppressing your anger and aggression instead of consciously acknowledging them',
 'to turn into a monster in your dream suggests that you are becoming someone who you are ashamed of or someone who you longer recognized or the dream may also be showing your true character perhaps you need to change your attitude or ways',
 'to dream that you are holding or expressing anger symbolizes frustrations and disappointments in your self you tend to repress your negative emotions or project your anger onto others you need to look within yourself',
 'to dream that you kill a monster means that you will successfully overcome your rivals and advance to a higher position',
 'to see someone else crying in your dream may be a projection of your own feelings onto someone else if you do not cry in your waking life then seei

In [12]:
# Initialize response Model
pipe = pipeline(
    "text-generation",
    model='microsoft/Phi-3-mini-4k-instruct',
    device='mps'
    )

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [13]:
# Formulate responses

# Define prompt template
prompt = f'You are interpreting a dream submitted by the user. The original dream text is :{dreamer_text}. The main interpretations for the dream are {interpretations[:5]}. You give a summary of these interpretations, strictly using these interpretations only and not adding any new idea, in less than 100 words.'
messages = [
        {'role': 'system', 'content': prompt},
    ]

# Query model
output = pipe(messages)
output[0]['generated_text'][-1]['content']


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


' The dream suggests anger and frustration, possibly from suppressed waking life emotions. The transformation into a monster symbolizes an unrecognizable or shameful aspect of oneself, indicating a need for change. Anger expressed in the dream may represent repressed emotions, and witnessing a loved one crying might reflect personal avoidance of vulnerability.'